# Quick check: the headline result in under 5 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skape007-ult/causal-probe/blob/main/repro/quickcheck.ipynb)

CPU only. No model weights and no GPU. Downloads the official Corr2Cause files (about 385 MB) from Hugging Face at
the revision the project pinned, recomputes the numbers below **by calling the repository's own code**
(`src/causal_probe/` and the two analysis scripts that produced them), and prints each next to the value committed in `results/`.

| Step | What is recomputed | Committed in |
|---|---|---|
| 1 | **Claim 1 headline.** A classifier that sees only the hypothesis, never the premise, fitted on the official train split, scored on the official test split: AUROC 0.895 [0.863, 0.919] | `results/benchmark_baselines.json` |
| 2 | **Debiased pool: the known shortcuts are removed.** Hypothesis-only, premise-only and additive whole-prompt classifiers are at chance | `results/benchmark_baselines.json` |
| 3 | **Debiased pool: prompt length still leaks within a hypothesis type** (and, for reference, claim 2's model-free "lookup", 0.765) | `results/probe_content_*_c2c_debiased.json` |

Where the computation sits inside a repository function together with other baselines, the notebook re-runs only the
lines needed and asserts that each of them occurs verbatim in that function; the data loading, pool construction, folds,
classifiers and bootstrap are called directly from the repository.

Every check passes only if the recomputed value equals the committed value when both are rounded to 3 decimals,
the precision used in `RESULTS.md` and in the report's tables (`report/tables/`). The last cell prints the overall verdict and the run time.

*Run it:* in Colab, **Runtime → Run all**. Locally, from a checkout of the repository:
`jupyter nbconvert --to notebook --execute repro/quickcheck.ipynb --output quickcheck_run.ipynb`
(needs `numpy`, `scipy`, `scikit-learn==1.9.1`, `joblib`, `huggingface_hub`).

*One version pin matters.* The prompt-length model in step 3 is gradient boosting on a single continuous feature, and its
output changed in scikit-learn 1.9.0: releases 1.6.1, 1.7.2 and 1.8.0 give accuracy 0.5561 instead of the committed 0.5625
(printed 0.562; AUROC 0.571 either way, and the interval stays above 0.5). Every other number here was the same when the whole
notebook was run under scikit-learn 1.6.1 and 1.9.1.
So the setup cell installs scikit-learn 1.9.1, the version in `requirements.txt`, when it runs in Colab, and warns
otherwise.

In [1]:
# Setup: find (or clone) the repository, and confirm that the code and results used here are exactly those of v1.0.1.
import os, subprocess, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, inside huggingface_hub; harmless

T_START = time.time()
REPO_URL = "https://github.com/skape007-ult/causal-probe"
FROZEN = ["src", "scripts", "results"]          # the v1.0 code and committed results this notebook checks against


def sh(*cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)


ROOT = next((p.resolve() for p in (Path("."), Path("..")) if (p / "src" / "causal_probe" / "data.py").exists()), None)
if ROOT is None:                                  # e.g. Colab: nothing checked out yet
    ROOT = Path("causal-probe").resolve()
    if not (ROOT / "src" / "causal_probe").exists():
        r = sh("git", "clone", "--quiet", "--depth", "1", REPO_URL, str(ROOT))
        assert r.returncode == 0, r.stderr
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

head = sh("git", "rev-parse", "--short", "HEAD").stdout.strip()
origin = sh("git", "remote", "get-url", "origin").stdout.strip() or REPO_URL
sh("git", "fetch", "--quiet", "--depth", "1", "origin", "tag", "v1.0.1")
diff = sh("git", "diff", "--quiet", "v1.0.1", "HEAD", "--", *FROZEN)
FROZEN_OK = diff.returncode == 0
print(f"repository: {origin} at commit {head}")
print(f"src/, scripts/ and results/ identical to tag v1.0.1: {'yes' if FROZEN_OK else 'NO (' + (diff.stderr.strip() or 'they differ') + ')'}")

# scikit-learn as pinned in requirements.txt (see the note above). Colab: installed here, before anything imports it.
SKLEARN_PIN = "1.9.1"
from importlib.metadata import version as _version, PackageNotFoundError
try:
    _have = _version("scikit-learn")
except PackageNotFoundError:
    _have = None
if _have != SKLEARN_PIN:
    if "google.colab" in sys.modules and "sklearn" not in sys.modules:
        r = sh(sys.executable, "-m", "pip", "install", "--quiet", f"scikit-learn=={SKLEARN_PIN}")
        assert r.returncode == 0, r.stderr
        print(f"installed scikit-learn {SKLEARN_PIN} (Colab had {_have})")
    else:
        print(f"WARNING: scikit-learn {_have} is installed; the committed results used {SKLEARN_PIN}. Step 3's prompt-length "
              f"accuracy and interval depend on it. Install scikit-learn=={SKLEARN_PIN} (and restart the kernel) to check them.")

import numpy, sklearn, joblib, huggingface_hub
print(f"python {sys.version.split()[0]}  numpy {numpy.__version__}  scikit-learn {sklearn.__version__}  "
      f"joblib {joblib.__version__}  huggingface_hub {huggingface_hub.__version__}  "
      f"CPUs available {len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count()}")

repository: https://github.com/skape007-ult/causal-probe at commit 84080ee
src/, scripts/ and results/ identical to tag v1.0.1: yes


python 3.12.3  numpy 2.5.4  scikit-learn 1.9.1  joblib 1.6.0  huggingface_hub 2.4.0  CPUs available 2


In [2]:
# Helpers: load the repository's analysis scripts as modules, read committed values, record checks.
import importlib.util, inspect, json, types
import numpy as np


def load_script(path, name):
    spec = importlib.util.spec_from_file_location(name, ROOT / path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def uses_repo_code(obj, snippets, where):
    """Every line this notebook re-runs from inside a repository function is asserted to appear in it verbatim."""
    src = inspect.getsource(obj)
    missing = [s for s in snippets if s not in src]
    assert not missing, f"not found verbatim in {where}: {missing}"
    print(f"  (each of the {len(snippets)} repository lines re-run below occurs verbatim in {where}; checked)")


def committed(path, *keys):
    d = json.loads((ROOT / path).read_text())
    for k in keys:
        d = d[k]
    return d


CHECKS = []
STEP = "setup"


def check(label, new, old, source):
    ok = f"{new:.3f}" == f"{old:.3f}"
    CHECKS.append((STEP, label, new, old, ok, source))
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<70} recomputed {new:.3f}   committed {old:.3f}   (|diff| {abs(new - old):.1e})")


def claim(label, ok):
    CHECKS.append((STEP, label, None, None, bool(ok), "derived"))
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")


claim("src/, scripts/ and results/ are those of tag v1.0.1", FROZEN_OK)

from causal_probe import data, probes, stats
from causal_probe.config import PROBE_SEEDS
dl = load_script("scripts/download_data.py", "download_data")
bb = load_script("scripts/14_benchmark_baselines.py", "benchmark_baselines")
# scripts/05b_probe_content.py imports causal_probe.cache (activation caches, needs torch) but uses it only inside
# main(), which is not called here. A placeholder module lets its text-only functions be imported without torch.
sys.modules.setdefault("causal_probe.cache", types.ModuleType("causal_probe.cache"))
pc = load_script("scripts/05b_probe_content.py", "probe_content")
print("loaded: src/causal_probe, scripts/download_data.py, scripts/14_benchmark_baselines.py, scripts/05b_probe_content.py")

  PASS  src/, scripts/ and results/ are those of tag v1.0.1


loaded: src/causal_probe, scripts/download_data.py, scripts/14_benchmark_baselines.py, scripts/05b_probe_content.py


## Step 0. Data: the official Corr2Cause files from Hugging Face

`causal-nlp/corr2cause` at the revision pinned in `scripts/download_data.py`, md5-checked against the sums listed there.
Only `train.csv` (to fit) and `test.csv` (to score) are needed.

In [3]:
import shutil
from huggingface_hub import hf_hub_download

t0 = time.time()
for repo, rev, name, local, want in dl.FILES:
    if name not in ("train.csv", "test.csv"):
        continue
    dst = dl.DATA_RAW / local
    if not (dst.exists() and dl.md5(dst) == want):
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(hf_hub_download(repo, name, repo_type="dataset", revision=rev), dst)
    got = dl.md5(dst)
    assert got == want, f"{local}: md5 {got} != {want}"
    print(f"  ok  {repo}@{rev[:8]}  {name:<9}  {dst.stat().st_size / 1e6:6.1f} MB  md5 {got}")

train, test = data.load_c2c("train"), data.load_c2c("test")
print(f"  train: {len(train)} items   test: {len(test)} items, {len({i['group'] for i in test})} premises   ({time.time() - t0:.0f} s)")

  ok  causal-nlp/corr2cause@42ba12c7  train.csv   383.4 MB  md5 bee2c98caa50f69bee54318d35f0ace4
  ok  causal-nlp/corr2cause@42ba12c7  test.csv      1.2 MB  md5 c318d19ddc585bb4310084e2ce2d241f


  train: 205734 items   test: 1162 items, 21 premises   (6 s)


## Step 1. Claim 1: the hypothesis alone ranks valid above invalid test items

The classifier is the repository's `hypothesis_only` baseline from `c2c_official` in
`scripts/14_benchmark_baselines.py`: P(valid | exact hypothesis string), estimated on the official train split, applied to the
test split; strings unseen in train get the train base rate. It never sees the premise. The interval is the repository's
cluster bootstrap over the 21 test premises (`metrics`, 2000 resamples).

In [4]:
t0 = time.time(); STEP = "1 claim 1"
uses_repo_code(bb.c2c_official, [
    'ytr = np.array([i["label"] for i in tr])',
    'keys = np.array([i["query"] for i in tr])',
    'hyp_prior = {k: float(ytr[keys == k].mean()) for k in np.unique(keys)}',
    'base = float(ytr.mean())',
    'y = np.array([i["label"] for i in te]); g = np.array([i["group"] for i in te])',
    'hp = np.array([hyp_prior.get(i["query"], base) for i in te])',
    '"hypothesis_only": metrics(y, logit(hp), g)',
], "c2c_official() in scripts/14_benchmark_baselines.py")
tr, te = train, test
ytr = np.array([i["label"] for i in tr])
keys = np.array([i["query"] for i in tr])
hyp_prior = {k: float(ytr[keys == k].mean()) for k in np.unique(keys)}
base = float(ytr.mean())
y = np.array([i["label"] for i in te]); g = np.array([i["group"] for i in te])
hp = np.array([hyp_prior.get(i["query"], base) for i in te])
m = bb.metrics(y, bb.logit(hp), g)

SRC = "results/benchmark_baselines.json"
old = committed(SRC, "corr2cause_official_splits", "test", "hypothesis_only")
print(f"\nHypothesis-only classifier, official test split ({m['n']} items, {m['n_clusters']} premises), fit on {len(tr)} train items:")
check("hypothesis-only, official test split: AUROC", m["auroc"]["est"], old["auroc"]["est"], SRC)
check("hypothesis-only, official test split: AUROC, 95% CI lower", m["auroc"]["lo"], old["auroc"]["lo"], SRC)
check("hypothesis-only, official test split: AUROC, 95% CI upper", m["auroc"]["hi"], old["auroc"]["hi"], SRC)
check("hypothesis-only, official test split: accuracy", m["accuracy"]["est"], old["accuracy"]["est"], SRC)
print(f"  ({time.time() - t0:.0f} s)")

  (each of the 7 repository lines re-run below occurs verbatim in c2c_official() in scripts/14_benchmark_baselines.py; checked)



Hypothesis-only classifier, official test split (1162 items, 21 premises), fit on 205734 train items:
  PASS  hypothesis-only, official test split: AUROC                            recomputed 0.895   committed 0.895   (|diff| 0.0e+00)
  PASS  hypothesis-only, official test split: AUROC, 95% CI lower              recomputed 0.863   committed 0.863   (|diff| 0.0e+00)
  PASS  hypothesis-only, official test split: AUROC, 95% CI upper              recomputed 0.919   committed 0.919   (|diff| 0.0e+00)
  PASS  hypothesis-only, official test split: accuracy                         recomputed 0.855   committed 0.855   (|diff| 0.0e+00)
  (9 s)


## Step 2. The debiased pool: the known shortcuts are removed

Claims 2 and 3 use a *debiased pool* of 2,400 train items (`c2c_debiased_pool` in `src/causal_probe/data.py`): only the
48 hypothesis strings that occur with both labels; 25 blocks, and inside each block every string appears once valid and once
invalid; every premise is used once, with one valid and one invalid hypothesis. Cross-validation folds are unions of whole
blocks. So a classifier that sees only the hypothesis, or only the premise, or adds up evidence from the two
(a linear bag-of-words of the whole prompt), has nothing to learn.

Below: the construction is checked directly, then the four text-only classifiers are refitted exactly as in `cv_pool`
(`scripts/14_benchmark_baselines.py`: premise-grouped 5-fold cross-validation, 3 seeds, scores averaged over seeds).

In [5]:
t0 = time.time(); STEP = "2 shortcuts"
from collections import Counter
items = data.c2c_debiased_pool(train)          # the same call, with the same default seed and size, that data.get_dataset("c2c_debiased") makes
del train
y = np.array([i["label"] for i in items]); g = np.array([i["group"] for i in items])
arr = np.array(items, dtype=object)
cell = Counter((i["block"], i["query"], i["label"]) for i in items)
prem = Counter((i["group"], i["label"]) for i in items)
print(f"Debiased pool: {len(items)} items, {len({i['query'] for i in items})} hypothesis strings, "
      f"{len(set(g))} premises, {len({i['block'] for i in items})} blocks")
claim("construction: every string once valid and once invalid in every block", set(cell.values()) == {1} and len(cell) == 2 * 25 * 48)
claim("construction: every premise has exactly one valid and one invalid item", set(prem.values()) == {1} and len(prem) == 2 * len(set(g)))

uses_repo_code(bb.cv_pool, [
    'seed_folds = [probes.folds_for(items, sd) for sd in PROBE_SEEDS]',
    'return np.mean([fn(f) for f in seed_folds], 0)',
    's[te] = premise_tfidf_fit_predict(list(arr[tr]), y[tr], list(arr[te]))',
    '"hypothesis_only": mean_oof(lambda f: probes.hyp_prior_oof(items, y, f))',
    '"hypothesis_only_tfidf": mean_oof(lambda f: probes.tfidf_query_oof(items, y, f))',
    '"premise_only": mean_oof(premise_oof)',
    '"full_text_tfidf": mean_oof(lambda f: probes.tfidf_oof(items, y, f, C=1.0))',
], "cv_pool() in scripts/14_benchmark_baselines.py")
uses_repo_code(bb.metrics, [
    '"accuracy": stats.bootstrap_ci(lambda a, s: stats.accuracy(a, pred(s)), (y, score), groups)',
    '"auroc": stats.bootstrap_ci(stats.auroc, (y, score), groups)',
], "metrics() in scripts/14_benchmark_baselines.py (only its accuracy and AUROC entries are computed here)")
seed_folds = [probes.folds_for(items, sd) for sd in PROBE_SEEDS]
mean_oof = lambda fn: np.mean([fn(f) for f in seed_folds], 0)


def premise_oof(folds):
    s = np.zeros(len(y), np.float32)
    for tr, te in folds:
        s[te] = bb.premise_tfidf_fit_predict(list(arr[tr]), y[tr], list(arr[te]))
    return s


scores = {"hypothesis_only": mean_oof(lambda f: probes.hyp_prior_oof(items, y, f)),
          "hypothesis_only_tfidf": mean_oof(lambda f: probes.tfidf_query_oof(items, y, f)),
          "premise_only": mean_oof(premise_oof),
          "full_text_tfidf": mean_oof(lambda f: probes.tfidf_oof(items, y, f, C=1.0))}
names = {"hypothesis_only": "hypothesis only (string prior)", "hypothesis_only_tfidf": "hypothesis only (TF-IDF)",
         "premise_only": "premise only (TF-IDF)", "full_text_tfidf": "whole prompt, additive (linear TF-IDF)"}
SRC = "results/benchmark_baselines.json"
print(f"\nText-only classifiers on the debiased pool (all {len(y)} items, out of fold):")
for k, s in scores.items():
    old = committed(SRC, "corr2cause_debiased_pool", "baselines", k)
    acc = stats.bootstrap_ci(lambda a, s_: stats.accuracy(a, (s_ > 0).astype(int)), (y, s), g)
    auc = stats.bootstrap_ci(stats.auroc, (y, s), g)
    check(f"{names[k]}: AUROC", auc["est"], old["auroc"]["est"], SRC)
    check(f"{names[k]}: accuracy", acc["est"], old["accuracy"]["est"], SRC)
    claim(f"{names[k]}: at chance (AUROC 95% CI [{auc['lo']:.3f}, {auc['hi']:.3f}] contains 0.5)", auc["lo"] <= 0.5 <= auc["hi"])
print(f"  ({time.time() - t0:.0f} s)")

Debiased pool: 2400 items, 48 hypothesis strings, 1200 premises, 25 blocks
  PASS  construction: every string once valid and once invalid in every block
  PASS  construction: every premise has exactly one valid and one invalid item
  (each of the 7 repository lines re-run below occurs verbatim in cv_pool() in scripts/14_benchmark_baselines.py; checked)
  (each of the 2 repository lines re-run below occurs verbatim in metrics() in scripts/14_benchmark_baselines.py (only its accuracy and AUROC entries are computed here); checked)



Text-only classifiers on the debiased pool (all 2400 items, out of fold):


  PASS  hypothesis only (string prior): AUROC                                  recomputed 0.500   committed 0.500   (|diff| 0.0e+00)
  PASS  hypothesis only (string prior): accuracy                               recomputed 0.500   committed 0.500   (|diff| 0.0e+00)
  PASS  hypothesis only (string prior): at chance (AUROC 95% CI [0.500, 0.500] contains 0.5)


  PASS  hypothesis only (TF-IDF): AUROC                                        recomputed 0.500   committed 0.500   (|diff| 0.0e+00)
  PASS  hypothesis only (TF-IDF): accuracy                                     recomputed 0.500   committed 0.500   (|diff| 0.0e+00)
  PASS  hypothesis only (TF-IDF): at chance (AUROC 95% CI [0.500, 0.500] contains 0.5)


  PASS  premise only (TF-IDF): AUROC                                           recomputed 0.500   committed 0.500   (|diff| 0.0e+00)
  PASS  premise only (TF-IDF): accuracy                                        recomputed 0.500   committed 0.500   (|diff| 0.0e+00)
  PASS  premise only (TF-IDF): at chance (AUROC 95% CI [0.500, 0.500] contains 0.5)


  PASS  whole prompt, additive (linear TF-IDF): AUROC                          recomputed 0.512   committed 0.512   (|diff| 0.0e+00)
  PASS  whole prompt, additive (linear TF-IDF): accuracy                       recomputed 0.495   committed 0.495   (|diff| 0.0e+00)
  PASS  whole prompt, additive (linear TF-IDF): at chance (AUROC 95% CI [0.491, 0.535] contains 0.5)
  (29 s)


## Step 3. What still leaks: prompt length within a hypothesis type

The debiased pool balances strings and premises, not lengths. `scripts/05b_probe_content.py` fits, for each of the five
hypothesis templates separately, a gradient-boosting classifier on one feature: the number of characters of premise plus
hypothesis (`template_x_prompt_length`). It is scored on the report half of the pool (1,248 items; the half never used to choose
layers or settings), with the script's cluster bootstrap (1000 resamples). If its AUROC interval lies above 0.5, prompt length
still carries label information within a hypothesis type. That is the caveat in claim 2's wording. (This model is the one that depends on the scikit-learn version; see the note at the top.)

For reference, the same per-template model on the premise statements that name the two queried variables
(`template_x_statements_naming_the_pair`, "the lookup") gives claim 2's lookup accuracy, 0.765.

These rows use no model, so the script writes the same values into every model's result file. The check reads them from the
Qwen2.5-3B file and confirms that all six files agree.

In [6]:
t0 = time.time(); STEP = "3 length"
uses_repo_code(pc.main, [
    'rep = ~data.selection_mask(items); sel = ~rep',
    'tmpl = np.array([i["template"] for i in items])',
    'F = probes.structural_features(items)',
    'Fcounts, Fpair, Fvar = F[:, 11:15], F[:, 0:6]',
    'seed_folds = [probes.folds_for(items, sd) for sd in PROBE_SEEDS]',
    'acc = lambda t, s: stats.accuracy(t, (s > 0).astype(int))',
    'groups[rep if m is None else m], n_boot=1000)',
    'mean_oof = lambda comp: np.mean([oof(comp, f, y) for f in seed_folds], 0)',
    'plen = np.array([len(i["context"]) + len(i["query"]) for i in items], float)[:, None]',
    '["template_x_prompt_length"] = mean_oof(by_key_gbm(tmpl, plen))',
    '"template_x_statements_naming_the_pair": mean_oof(by_key_gbm(tmpl, Fpair))',
    'return {"accuracy": ci(acc, y, s), "auroc": ci(stats.auroc, y, s),',
], "main() in scripts/05b_probe_content.py")
groups = g
rep = ~data.selection_mask(items)
tmpl = np.array([i["template"] for i in items])
F = probes.structural_features(items)
Fpair = F[:, 0:6]
plen = np.array([len(i["context"]) + len(i["query"]) for i in items], float)[:, None]
mean_oof = lambda comp: np.mean([pc.oof(comp, f, y) for f in seed_folds], 0)
acc = lambda t, s: stats.accuracy(t, (s > 0).astype(int))
ci = lambda fn, *arrs: stats.bootstrap_ci(fn, tuple(x[rep] for x in arrs), groups[rep], n_boot=1000)

MODELS6 = ["qwen0.5b", "qwen1.5b", "gemma2b", "qwen3b", "qwen7b-4bit", "gemma9b"]
SRC = "results/probe_content_qwen3b_c2c_debiased.json"
rows = {"template_x_prompt_length": ("A_no_binding_interaction", "prompt length, per hypothesis type", plen),
        "template_x_statements_naming_the_pair": ("B_binding_text_reference", "lookup of statements naming the pair (claim 2)", Fpair)}
print(f"Per-template classifiers on the debiased pool, report half ({int(rep.sum())} items):")
for k, (grp, label, X) in rows.items():
    allsame = len({json.dumps(committed(f"results/probe_content_{mm}_c2c_debiased.json", "baselines", grp, k)[q]["est"])
                   for mm in MODELS6 for q in ("accuracy", "auroc")}) == 2
    old = committed(SRC, "baselines", grp, k)
    s = mean_oof(pc.by_key_gbm(tmpl, X))
    a_, u_ = ci(acc, y, s), ci(stats.auroc, y, s)
    claim(f"{label}: committed value identical in all six models' files", allsame)
    check(f"{label}: AUROC", u_["est"], old["auroc"]["est"], SRC)
    if k == "template_x_prompt_length":
        check(f"{label}: AUROC, 95% CI lower", u_["lo"], old["auroc"]["lo"], SRC)
        check(f"{label}: AUROC, 95% CI upper", u_["hi"], old["auroc"]["hi"], SRC)
    check(f"{label}: accuracy", a_["est"], old["accuracy"]["est"], SRC)
    if k == "template_x_prompt_length":
        claim(f"{label}: still leaks (AUROC 95% CI [{u_['lo']:.3f}, {u_['hi']:.3f}] lies above 0.5)", u_["lo"] > 0.5)
print(f"  ({time.time() - t0:.0f} s)")

  (each of the 12 repository lines re-run below occurs verbatim in main() in scripts/05b_probe_content.py; checked)
Per-template classifiers on the debiased pool, report half (1248 items):


  PASS  prompt length, per hypothesis type: committed value identical in all six models' files
  PASS  prompt length, per hypothesis type: AUROC                              recomputed 0.571   committed 0.571   (|diff| 0.0e+00)
  PASS  prompt length, per hypothesis type: AUROC, 95% CI lower                recomputed 0.539   committed 0.539   (|diff| 0.0e+00)
  PASS  prompt length, per hypothesis type: AUROC, 95% CI upper                recomputed 0.603   committed 0.603   (|diff| 0.0e+00)
  PASS  prompt length, per hypothesis type: accuracy                           recomputed 0.562   committed 0.562   (|diff| 0.0e+00)
  PASS  prompt length, per hypothesis type: still leaks (AUROC 95% CI [0.539, 0.603] lies above 0.5)


  PASS  lookup of statements naming the pair (claim 2): committed value identical in all six models' files
  PASS  lookup of statements naming the pair (claim 2): AUROC                  recomputed 0.834   committed 0.834   (|diff| 0.0e+00)
  PASS  lookup of statements naming the pair (claim 2): accuracy               recomputed 0.765   committed 0.765   (|diff| 0.0e+00)
  (13 s)


## Verdict

In [7]:
n_pass = sum(c[4] for c in CHECKS)
print(f"{'':4}  {'step':<11} {'check':<96} {'recomputed':>10} {'committed':>10}")
for step, label, new, old, ok, src in CHECKS:
    nums = f"{new:>10.3f} {old:>10.3f}" if new is not None else f"{'':>10} {'':>10}"
    print(f"{'PASS' if ok else 'FAIL'}  {step:<11} {label[:96]:<96} {nums}")
print()
print(f"{n_pass} of {len(CHECKS)} checks pass.  " + ("ALL CHECKS PASS." if n_pass == len(CHECKS) else "SOME CHECKS FAIL: see the FAIL lines above."))
if sklearn.__version__ != SKLEARN_PIN and n_pass < len(CHECKS):
    print(f"Note: scikit-learn {sklearn.__version__} is installed, not {SKLEARN_PIN}; see the version note at the top.")
print(f"Wall-clock time for the whole notebook: {time.time() - T_START:.0f} s")

      step        check                                                                                            recomputed  committed
PASS  setup       src/, scripts/ and results/ are those of tag v1.0.1                                                                   
PASS  1 claim 1   hypothesis-only, official test split: AUROC                                                           0.895      0.895
PASS  1 claim 1   hypothesis-only, official test split: AUROC, 95% CI lower                                             0.863      0.863
PASS  1 claim 1   hypothesis-only, official test split: AUROC, 95% CI upper                                             0.919      0.919
PASS  1 claim 1   hypothesis-only, official test split: accuracy                                                        0.855      0.855
PASS  2 shortcuts construction: every string once valid and once invalid in every block                                                 
PASS  2 shortcuts construction: every pre